# 01 — Data Validation

**Purpose:** Verify the raw Walmart M5 files before any transformation or analysis.
Every finding in this notebook comes from code run on the actual files.

**How to run:** In VS Code, click **Select Kernel → Python Environments → `.venv` (Python 3.13.2)**,
then run cells from top to bottom.

**Rules**
- Files in `data/raw/` are read only — never modified.
- Outputs stay small (summaries and `.head()`), so the notebook stays light on GitHub.
- Check logic lives in `src/validate_data.py`; this notebook calls it and explains results.

## Step 1 — Locate the project folder

VS Code runs notebooks from the `notebooks/` folder. To import from `src/`, we find the
project folder (the one containing `src/` and `requirements.txt`) and add it to Python's
import path. No absolute paths are hardcoded, so this works on any machine.

`%autoreload 2` reloads edited `src/` code automatically, without restarting the kernel.

In [5]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Walk up from `start` to the folder containing src/ and requirements.txt."""
    for folder in [start, *start.parents]:
        if (folder / "src").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError(
        "Project folder not found. Open this notebook from inside the RetailIQ project."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project folder:", PROJECT_ROOT.name)
print("Notebook running from:", Path.cwd().resolve().relative_to(PROJECT_ROOT) or ".")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project folder: retailiq-ai-business-intelligence
Notebook running from: notebooks


## Step 2 — Verify the Python environment

The notebook must run on the project's `.venv` (Python 3.13.2, pinned packages).
If this cell fails, the wrong kernel is selected — use **Select Kernel** to choose `.venv`.

In [6]:
import pandas as pd

expected_env = (PROJECT_ROOT / ".venv").resolve()
active_env = Path(sys.prefix).resolve()

if active_env != expected_env:
    raise RuntimeError(
        "Wrong kernel: this notebook is not running on the project's .venv. "
        "Use Select Kernel → .venv (Python 3.13.2)."
    )

print("Interpreter:", Path(sys.executable).relative_to(PROJECT_ROOT))
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

Interpreter: .venv/bin/python
Python: 3.13.2
pandas: 3.1.1


## Step 3 — Import the validation module

Confirms the notebook can use code from `src/validate_data.py`. Importing it only loads
its functions and settings; it does not run any checks.

In [7]:
from src import validate_data

print("Module:", Path(validate_data.__file__).relative_to(PROJECT_ROOT))
print("Raw data folder:", validate_data.RAW_DIR.relative_to(PROJECT_ROOT))
print()
for name in validate_data.EXPECTED_FILES:
    status = "found" if (validate_data.RAW_DIR / name).is_file() else "MISSING"
    print(f"  {name:<28} {status}")

Module: src/validate_data.py
Raw data folder: data/raw

  calendar.csv                 found
  sell_prices.csv              found
  sales_train_validation.csv   found
  sales_train_evaluation.csv   found
  sample_submission.csv        found


## Step 4 — Preview `calendar.csv`

A first look at the smallest file. We load it through the module's `RAW_DIR`, which also
confirms the notebook and the script read from the same folder.

In [8]:
calendar = pd.read_csv(validate_data.RAW_DIR / "calendar.csv")
print(f"Shape: {calendar.shape[0]:,} rows × {calendar.shape[1]} columns")
calendar.head()

Shape: 1,969 rows × 14 columns


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


# Group 2 — Calendar validation

`calendar.csv` links every day key (`d_1`, `d_2`, …) to a real date, a Walmart week
(`wm_yr_wk`), SNAP days, and events. Every later join depends on it, so we check it first.

Each check returns PASS (succeeded), INFO (a measured fact), or FAIL (a real problem).
Problems are reported as FAIL results rather than errors, so every check always runs.
The logic lives in `src/validate_data.py`; here we run it and look at the results.

In [9]:
calendar = validate_data.load_calendar()
print(f"Loaded {len(calendar):,} rows; date column type: {calendar['date'].dtype}")

Loaded 1,969 rows; date column type: datetime64[us]


## 2.1 Unique dates
Every row should be a different, valid date (`YYYY-MM-DD`). A duplicate date would
double-count sales when we join on it. Invalid or missing dates are FAIL, and the date
range is calculated from valid dates only.

In [10]:
validate_data.as_table(validate_data.check_unique_dates(calendar))

,level,check,message
0,PASS,unique dates,all dates parse as YYYY-MM-DD
1,PASS,unique dates,every valid date appears exactly once
2,INFO,unique dates,range of valid dates: 2011-01-29 to 2016-06-19


## 2.2 Date continuity
Dates should run day by day with no gaps. A missing day would silently break trend
charts and seasonal comparisons. If any date is invalid, continuity is reported as
FAIL (not checked), because gaps around an unreadable date are ambiguous.

In [11]:
validate_data.as_table(validate_data.check_date_continuity(calendar))

,level,check,message
0,PASS,date continuity,dates are in ascending order in the file
1,PASS,date continuity,"1,969 consecutive daily dates, no gaps"


## 2.3 Day-key sequence
The sales files use day keys (`d_1` … `d_N`) as column names instead of dates. They must
be unique and run in order alongside the dates, or sales would be matched to the wrong day.

In [12]:
validate_data.as_table(validate_data.check_day_keys(calendar))

,level,check,message
0,PASS,day keys,every day key is unique
1,PASS,day keys,day keys run d_1 to d_1969 in date order


## 2.4 Week-key completeness
Prices are recorded per Walmart week (`wm_yr_wk`), so every day needs a valid, whole-number
week key. Each week should be one unbroken run of 7 days; only the first or last week may
be partial. If any key is missing or invalid, the structure checks are skipped.

In [13]:
display(validate_data.as_table(validate_data.check_week_keys(calendar)))
validate_data.calendar_week_sizes(calendar)

,level,check,message
0,PASS,week keys,every day has a whole-number week key
1,PASS,week keys,week keys never decrease over time
2,PASS,week keys,each week key covers one unbroken run of days
3,INFO,week keys,282 week keys; 281 have 7 days
4,PASS,week keys,all weeks inside the range have 7 days
5,INFO,week keys,last week (11621) is partial: 2 days


,days_in_week,weeks
0,2,1
1,7,281


## 2.5 SNAP values
SNAP flags mark days when food-assistance benefits could be used in each state. They
must be 0 or 1 with no gaps, because we'll compare sales on SNAP vs. non-SNAP days.

In [14]:
validate_data.as_table(validate_data.check_snap(calendar))

,level,check,message
0,PASS,SNAP values,"snap_CA: only 0 or 1, none missing"
1,INFO,SNAP values,"snap_CA: 650 SNAP days of 1,969"
2,PASS,SNAP values,"snap_TX: only 0 or 1, none missing"
3,INFO,SNAP values,"snap_TX: 650 SNAP days of 1,969"
4,PASS,SNAP values,"snap_WI: only 0 or 1, none missing"
5,INFO,SNAP values,"snap_WI: 650 SNAP days of 1,969"


## 2.6 Missing event values
Event columns are expected to be blank on days without an event, so blanks alone are
INFO. Problems are an event name without a type (or the reverse), or a second event
without a first. These columns drive the holiday analysis later.

In [15]:
display(validate_data.as_table(validate_data.check_events(calendar)))
display(validate_data.calendar_missing_summary(calendar))
validate_data.calendar_event_types(calendar)

,level,check,message
0,INFO,event values,"event_name_1: blank on 1,807 of 1,969 days (91..."
1,PASS,event values,event_name_1 and event_type_1 are always fille...
2,INFO,event values,"event_name_2: blank on 1,964 of 1,969 days (99..."
3,PASS,event values,event_name_2 and event_type_2 are always fille...
4,PASS,event values,a second event only appears when a first event...
5,INFO,event values,"30 distinct event names; types: Cultural, Nati..."


,missing,missing_pct
date,0,0.0
wm_yr_wk,0,0.0
weekday,0,0.0
wday,0,0.0
month,0,0.0
year,0,0.0
d,0,0.0
event_name_1,1807,91.8
event_type_1,1807,91.8
event_name_2,1964,99.7


,event_type,occurrences
0,Religious,56
1,National,52
2,Cultural,41
3,Sporting,18


## 2.7 Calendar vs. sales coverage (the 28-day difference)
Group 1 showed the calendar has more days than the evaluation sales file has day columns.
Here we check that every sales day has a calendar date, and list exactly which calendar
days have no sales. Only the sales files' header lines are read.

The list is shown only if the coverage checks for the evaluation file pass; otherwise
`calendar_days_without_sales()` refuses and explains why, so an invalid header can never
produce a misleading table.

In [16]:
display(validate_data.as_table(validate_data.check_sales_day_coverage(calendar)))

try:
    no_sales = validate_data.calendar_days_without_sales(calendar)
except ValueError as exc:
    print("Days without sales not listed:", exc)
else:
    print(f"{len(no_sales):,} calendar days without sales in the evaluation file")
    display(no_sales)

,level,check,message
0,PASS,sales coverage,sales_train_validation.csv: every sales day co...
1,PASS,sales coverage,sales_train_validation.csv: day columns match ...
2,INFO,sales coverage,sales_train_validation.csv: sales cover 2011-0...
3,PASS,sales coverage,sales_train_evaluation.csv: every sales day co...
4,PASS,sales coverage,sales_train_evaluation.csv: day columns match ...
5,INFO,sales coverage,sales_train_evaluation.csv: sales cover 2011-0...


28 calendar days without sales in the evaluation file


,d,date,weekday,wm_yr_wk,event_name_1,event_name_2
1941,d_1942,2016-05-23,Monday,11617,NaN,NaN
1942,d_1943,2016-05-24,Tuesday,11617,NaN,NaN
1943,d_1944,2016-05-25,Wednesday,11617,NaN,NaN
1944,d_1945,2016-05-26,Thursday,11617,NaN,NaN
1945,d_1946,2016-05-27,Friday,11617,NaN,NaN
1946,d_1947,2016-05-28,Saturday,11618,NaN,NaN
1947,d_1948,2016-05-29,Sunday,11618,NaN,NaN
1948,d_1949,2016-05-30,Monday,11618,MemorialDay,NaN
1949,d_1950,2016-05-31,Tuesday,11618,NaN,NaN
1950,d_1951,2016-06-01,Wednesday,11618,NaN,NaN


## Group 2 summary
All Group 2 results together, with any failures listed.

In [17]:
group2 = validate_data.as_table(
    [r for check in validate_data.CALENDAR_CHECKS for r in check(calendar)]
)
display(group2["level"].value_counts())
group2[group2["level"] == "FAIL"]

level
PASS    20
INFO    11
Name: count, dtype: int64

,level,check,message


## Group 2 findings

**Result: 20 PASS, 11 INFO, 0 FAIL.** `calendar.csv` is structurally sound and can be
used as the date reference for all joins.

### Measured results
| Area | Result |
|---|---|
| Dates | 1,969 unique, valid dates, consecutive with no gaps: 2011-01-29 to 2016-06-19 |
| Day keys | Unique, running `d_1` to `d_1969` in date order |
| Week keys | 282 week keys, none missing; each covers one unbroken run of days. 281 weeks have 7 days; the last week (11621) has 2 |
| SNAP | `snap_CA`, `snap_TX`, `snap_WI` contain only 0/1, none missing; 650 SNAP days per state across the full calendar |
| Events | Blank on 91.8% of days (`event_name_1`) and 99.7% (`event_name_2`), meaning no event recorded. Names and types are always filled together. 30 distinct event names; occurrences by type: Religious 56, National 52, Cultural 41, Sporting 18 |

### Calendar days without observed sales
| Sales file | Day columns | Dates covered | Calendar days without sales |
|---|---|---|---|
| `sales_train_validation.csv` | `d_1`–`d_1913` | 2011-01-29 to 2016-04-24 | 56 |
| `sales_train_evaluation.csv` | `d_1`–`d_1941` | 2011-01-29 to 2016-05-22 | **28** |

In both files, every sales day has a calendar date, and the day columns match the
calendar's first N days in order. The 28 days without sales in the evaluation file are
`d_1942`–`d_1969` (2016-05-23 to 2016-06-19). They include five recorded events:
MemorialDay, NBAFinalsStart, Ramadan starts, NBAFinalsEnd and Father's day.

*External context (not measured here):* the M5 competition withheld actual sales for its
final 28 days, the period teams had to forecast. That explains why these days exist in
the calendar without sales.

### Implications for dashboard KPIs
- **Analysis window (provisional until Group 4 confirms the overlap between the two
  sales files):** 2011-01-29 to 2016-05-22 (1,941 days), from the evaluation file.
- **Observed historical sales KPIs** (and their trends and averages) exclude days
  `d_1942`–`d_1969`. These days stay in the calendar and may be useful for forecasting
  later.
- **No sales data ≠ zero sales:** joins start from the sales data and add calendar
  fields, so days without sales never enter as zeros and pull averages down.
- **Partial periods at the end of the window:**
  - Week 11617 has only 2 of its 7 days in the window (2016-05-21 and 2016-05-22).
  - May 2016 has 22 of its 31 days in the window.
  - Weekly and monthly charts must flag or exclude these, so they don't look like a
    sales drop.
- **Event and SNAP counts above cover the full calendar.** Counts inside the analysis
  window (for example, the 2016 MemorialDay and Father's day fall outside it) will be
  measured when building the holiday analysis.

### Still to confirm
Whether the validation file's 1,913 days are identical to the evaluation file's first
1,913 days. If so, only the evaluation file is needed (Group 4).